In [ ]:
import pickle

# Load the saved model
with open("bodo_tone_classifier.pkl", "rb") as f:
    model = pickle.load(f)

# Example test sentence and tonal word
sentence = "बासननि आखाय बायनायजों बेखौ हमनायाव गोब्राब जायो"
tonal_word = "आखाय"

# Combine the input the same way as during training
input_text = sentence + " [TONAL] " + tonal_word

# Predict the tone
predicted_tone = model.predict([input_text])[0]
print(f"Predicted tone for '{tonal_word}' in the sentence: {predicted_tone}")


Predicted tone for 'आखाय' in the sentence: H


/usr/local/lib/python3.11/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.1.3 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.1.3 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.1.3 when using version 1.6.1. This might lead to breaking c

In [ ]:
import pickle

# Load the saved model
with open("bodo_tone_classifier.pkl", "rb") as f:
    model = pickle.load(f)

# Take user input
sentence = input("Enter a sentence in Bodo: ").strip()
tonal_word = input("Enter the tonal word from the sentence: ").strip()

# Validate input
if not sentence or not tonal_word:
    print("Error: Both sentence and tonal word must be provided.")
else:
    # Combine the input in the same way as during training
    input_text = sentence + " [TONAL] " + tonal_word

    # Predict the tone
    predicted_tone = model.predict([input_text])[0]
    print(f"Predicted tone for '{tonal_word}' in the sentence: {predicted_tone}")

Enter a sentence in Bodo: बासननि आखाय बायनायजों बेखौ हमनायाव गोब्राब जायो
Enter the tonal word from the sentence: आखाय
Predicted tone for 'आखाय' in the sentence: H


In [ ]:
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load the saved model
with open("bodo_tone_classifier.pkl", "rb") as f:
    model = pickle.load(f)

# Load the original dataset
df = pd.read_csv("example_with_tone.csv")

# Drop missing values
df = df.dropna()

# Prepare features and labels
X = df.iloc[:, 0] + " [TONAL] " + df.iloc[:, 1]
y = df.iloc[:, 2]

# Split into training and testing sets (same way as before)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Predict on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy on test set: {accuracy:.2%}")

Model accuracy on test set: 58.70%


In [16]:
print(df.iloc[:, 2].value_counts())


H
L    233
H    226
Name: count, dtype: int64


In [17]:
X = df.iloc[:, 0].astype(str) + " [TONAL] " + df.iloc[:, 1].astype(str)


In [18]:
print(type(model))


<class 'sklearn.pipeline.Pipeline'>


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X_vec = vectorizer.fit_transform(X)


In [20]:
from sklearn.model_selection import GridSearchCV

params = {'C': [0.1, 1, 10]}
clf = GridSearchCV(LogisticRegression(), param_grid=params, cv=5)
clf.fit(X_train_vec, y_train)
print("Best parameters:", clf.best_params_)


NameError: name 'LogisticRegression' is not defined

In [21]:
from sklearn.linear_model import LogisticRegression


In [22]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

params = {'C': [0.1, 1, 10]}
clf = GridSearchCV(LogisticRegression(), param_grid=params, cv=5)
clf.fit(X_train_vec, y_train)
print("Best parameters:", clf.best_params_)


NameError: name 'X_train_vec' is not defined

In [24]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Rebuild pipeline (if needed)
pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', LogisticRegression())
])

# Define parameter grid
param_grid = {
    'vectorizer__ngram_range': [(1, 1), (1, 2)],
    'vectorizer__max_features': [1000, 5000],
    'classifier__C': [0.1, 1, 10]
}

# Perform grid search
grid = GridSearchCV(pipeline, param_grid=param_grid, cv=5)
grid.fit(X_train, y_train)  # Use raw text here

# Results
print("Best accuracy:", grid.best_score_)
print("Best parameters:", grid.best_params_)


Best accuracy: 0.5450573861532765
Best parameters: {'classifier__C': 1, 'vectorizer__max_features': 1000, 'vectorizer__ngram_range': (1, 2)}


In [25]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[26 13]
 [25 28]]
              precision    recall  f1-score   support

           H       0.51      0.67      0.58        39
           L       0.68      0.53      0.60        53

    accuracy                           0.59        92
   macro avg       0.60      0.60      0.59        92
weighted avg       0.61      0.59      0.59        92



In [26]:
with open("improved_bodo_tone_classifier.pkl", "wb") as f:
    pickle.dump(clf, f)


In [27]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report
import warnings

warnings.filterwarnings("ignore")  # Optional: Suppress warnings for cleaner output

# Load dataset
df = pd.read_excel("example_with_tone.xlsx")
df = df.dropna()

# Combine sentence and tonal word with clear marker
X = df.iloc[:, 0].astype(str) + " [TONAL] " + df.iloc[:, 1].astype(str)
y = df.iloc[:, 2]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create pipeline
pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Define hyperparameter grid
param_grid = {
    'vectorizer__ngram_range': [(1, 1), (1, 2)],
    'vectorizer__max_features': [1000, 5000],
    'classifier__C': [0.1, 1, 10],
    'classifier__class_weight': [None, 'balanced']
}

# Grid search
grid = GridSearchCV(pipeline, param_grid=param_grid, cv=5, n_jobs=-1)
grid.fit(X_train, y_train)

# Best model
print("✅ Best parameters found:", grid.best_params_)
print("✅ Best CV accuracy: {:.2f}%".format(grid.best_score_ * 100))

# Evaluate on test set
y_pred = grid.predict(X_test)

print("\n📊 Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred))

# Optional: Save the best model
import pickle
with open("improved_bodo_tone_classifier.pkl", "wb") as f:
    pickle.dump(grid.best_estimator_, f)


✅ Best parameters found: {'classifier__C': 1, 'classifier__class_weight': 'balanced', 'vectorizer__max_features': 1000, 'vectorizer__ngram_range': (1, 1)}
✅ Best CV accuracy: 59.14%

📊 Confusion Matrix:
[[26 19]
 [20 27]]

📋 Classification Report:
              precision    recall  f1-score   support

           H       0.57      0.58      0.57        45
           L       0.59      0.57      0.58        47

    accuracy                           0.58        92
   macro avg       0.58      0.58      0.58        92
weighted avg       0.58      0.58      0.58        92

